# Lechture simple d'un fichier txt en python


In [83]:
with open("../data/text_simples/intro_rag.txt", "r", encoding="utf-8") as f:
    texte= f.read()
print(texte)    

l'ingestion est la premiere étape d'un project RAG.
elle permet au modèlle de comprendre des données externes.
un bon Parsing améliore la qualité des réponses. 


# transformer le texte en Document

In [84]:
from langchain_core.documents import Document

In [85]:
doc = Document(page_content=texte, metadata={"source": "intro_rag.txt", "format": "txt"})

In [86]:
print(doc)

page_content='l'ingestion est la premiere étape d'un project RAG.
elle permet au modèlle de comprendre des données externes.
un bon Parsing améliore la qualité des réponses. ' metadata={'source': 'intro_rag.txt', 'format': 'txt'}


In [87]:
print("content:", doc.page_content[:80], "...")
print("metadata:", doc.metadata)

content: l'ingestion est la premiere étape d'un project RAG.
elle permet au modèlle de co ...
metadata: {'source': 'intro_rag.txt', 'format': 'txt'}


# untiliser un Textloader(LangChain)

In [88]:
from langchain_community.document_loaders import TextLoader


In [89]:
# creation du chargeur de texte
loader = TextLoader("../data/text_simples/intro_rag.txt", encoding="utf-8")

In [90]:
docs=loader.load()

In [91]:
type(docs),len(docs),

(list, 1)

In [92]:
docs[0]

Document(metadata={'source': '../data/text_simples/intro_rag.txt'}, page_content="l'ingestion est la premiere étape d'un project RAG.\nelle permet au modèlle de comprendre des données externes.\nun bon Parsing améliore la qualité des réponses. ")

In [93]:
docs[0].metadata

{'source': '../data/text_simples/intro_rag.txt'}

# creation d'un second fichier text

In [94]:
# creation d'un second fichier texte pour la demonstration 

sample_text = """
les fichiers texte sont la forme la plus simple de documents. Ils contiennent du texte brut, sans mise en forme ni structure complexe.
pour les projets réel, tu auras souvent des pdf, des fichiers Word, des pages web ou d'autres formats de documents.
l'idée est de tout standarniser sous forme de texte brut pour pouvoir les traiter de manière uniforme.

"""

with open("../data/text_simples/intro_rag_2.txt", "w", encoding="utf-8") as f:
    f.write(sample_text)
print("Nouveau fichier texte créé : intro_rag_2.txt")

Nouveau fichier texte créé : intro_rag_2.txt


# demonstration plusieurs fichiers texte

In [95]:
# exemple: chargement de plusieurs fichiers texte dans un dossier
from langchain_community.document_loaders import DirectoryLoader

multi_loader = DirectoryLoader("../data/text_simples", glob="*.txt", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"})
documents = multi_loader.load()
print(f"Nombre de documents chargés : {len(documents)}")
for d in documents:
    print("content:", d.page_content[:80], "...")
    print("metadata:", d.metadata)

Nombre de documents chargés : 3
content: Titre : Guide d'Utilisation – Module Ingestion & Nettoyage

Ce fichier texte ser ...
metadata: {'source': '../data/text_simples/texte_exemple_rag.txt'}
content: l'ingestion est la premiere étape d'un project RAG.
elle permet au modèlle de co ...
metadata: {'source': '../data/text_simples/intro_rag.txt'}
content: 
les fichiers texte sont la forme la plus simple de documents. Ils contiennent d ...
metadata: {'source': '../data/text_simples/intro_rag_2.txt'}


# Ajouter des métadonnées personnalées

In [96]:
# on reprend les données chargés précédemment 

for doc in documents:
    
    print("metadata:", doc.metadata)

metadata: {'source': '../data/text_simples/texte_exemple_rag.txt'}
metadata: {'source': '../data/text_simples/intro_rag.txt'}
metadata: {'source': '../data/text_simples/intro_rag_2.txt'}


In [97]:
# Ajout de metadonnées personnalisées à chaque document lors du chargement
for doc in documents:
    #on regarde le nom du fichier pour définir un therme    automatiquement
    if "intro" in doc.metadata["source"]:
        doc.metadata["theme"] = "introduction"
    elif "ingestion" in doc.metadata["source"]:
        doc.metadata["theme"] = "directoryloader"
    #on peut aussi ajouter d'autres infos namuellement

    doc.metadata["author"] = "eg"
    doc.metadata["date"] = "2024-06-10"
    doc.metadata["version"] = "1.0"
    

    
# print
for doc in documents:
    print("metadata:", doc.metadata)


metadata: {'source': '../data/text_simples/texte_exemple_rag.txt', 'author': 'eg', 'date': '2024-06-10', 'version': '1.0'}
metadata: {'source': '../data/text_simples/intro_rag.txt', 'theme': 'introduction', 'author': 'eg', 'date': '2024-06-10', 'version': '1.0'}
metadata: {'source': '../data/text_simples/intro_rag_2.txt', 'theme': 'introduction', 'author': 'eg', 'date': '2024-06-10', 'version': '1.0'}


In [174]:
from langchain_community.document_loaders import TextLoader
# charger le fichier exemple
loader = TextLoader("../data/text_simples/texte_exemple_rag.txt", encoding="utf-8")
documents = loader.load()
# afficher un apercu du texte brut
print(documents[0].page_content[:1000], "...")

Titre : Guide d'Utilisation – Module Ingestion & Nettoyage

Ce fichier texte sert d'exemple pour la démonstration de l'étape de nettoyage
dans le pipeline RAG. Il contient volontairement plusieurs éléments de "bruit"
afin d'observer l'effet du nettoyage :

-    Espaces multiples
- Retours    à   la   ligne
- Caractères superflus
- Sections vides


Voici un exemple de paragraphe mal formaté :

Le nettoyage du texte est une étape essentielle.
Il permet      d'améliorer    la    qualité des embeddings.
   Plus le    texte est propre,
mieux le modèle comprendra le contenu.


Section 2 : Exemples supplémentaires de bruit textuel

Les documents réels contiennent souvent   des artefacts issus
de la conversion PDF vers texte, comme par exemple :

***   ***   ***

1)  Des puces mal alignées
2)     Des numéros de page isolés
3) Des en-têtes répétés sur chaque page


-------------------------------------
Page 1 sur 12
-------------------------------------

Voici un paragraphe avec des tabulations

In [182]:
def clean_text(text):
   import re
   texte = re.sub(r'\s+', ' ', text)  # Remplace les espaces multiples par un seul espace
   texte = re.sub(r'[^\w\s]', '', texte)  # Supprime la ponctuation
   texte = texte.lower()  # Met le texte en minuscules
   texte = texte.strip()  # Supprime les espaces au début et à la fin
   return texte

In [183]:
def clean_document(documents):
    cleaned_docs = []

    for doc in documents:
        cleaned_content = clean_text(doc.page_content)
        cleaned_doc = Document(
            page_content=cleaned_content,
            metadata=doc.metadata.copy()
        )
        cleaned_docs.append(cleaned_doc)

    return cleaned_docs

In [184]:
cleaned_documents = clean_document(documents)


In [185]:
print("=== Nettoyage des documents terminé ===")
print(cleaned_documents[0].page_content[:1000], "...")

=== Nettoyage des documents terminé ===
titre  guide dutilisation  module ingestion  nettoyage ce fichier texte sert dexemple pour la démonstration de létape de nettoyage dans le pipeline rag il contient volontairement plusieurs éléments de bruit afin dobserver leffet du nettoyage   espaces multiples  retours à la ligne  caractères superflus  sections vides voici un exemple de paragraphe mal formaté  le nettoyage du texte est une étape essentielle il permet daméliorer la qualité des embeddings plus le texte est propre mieux le modèle comprendra le contenu section 2  exemples supplémentaires de bruit textuel les documents réels contiennent souvent des artefacts issus de la conversion pdf vers texte comme par exemple     1 des puces mal alignées 2 des numéros de page isolés 3 des entêtes répétés sur chaque page  page 1 sur 12  voici un paragraphe avec des tabulations mélangées à des espaces ainsi que des espaces multiples consécutifs ce paragraphe commence par une indentation inutile et 

# découper les documents(document Splitters)

# RecursiveCharacterTextSplitter

In [186]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from regex import splititer

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # Taille maximale de chaque chunk
    chunk_overlap=150,  # Chevauchement entre les chunks
    length_function=len,  # Fonction pour calculer la longueur du texte
    separators=["\n\n", "\n", ",", " ", ""]
)

In [187]:
chunks = splitter.split_documents(cleaned_documents)

In [189]:
# inspecter les chunks
print(f"Nombre de chunks créés : {len(chunks)}")
print(chunks[0].page_content)
print(chunks[1].page_content)
print(chunks[3].page_content)

Nombre de chunks créés : 14
titre  guide dutilisation  module ingestion  nettoyage ce fichier texte sert dexemple pour la démonstration de létape de nettoyage dans le pipeline rag il contient volontairement plusieurs éléments de bruit afin dobserver leffet du nettoyage   espaces multiples  retours à la ligne  caractères superflus  sections vides voici un exemple de paragraphe mal formaté  le nettoyage du texte est une étape essentielle il permet daméliorer la qualité des embeddings plus le texte est propre mieux le modèle comprendra le contenu section 2  exemples supplémentaires de bruit textuel les documents réels contiennent souvent des artefacts issus de la conversion pdf vers texte comme par exemple     1 des puces mal alignées 2 des numéros de page isolés 3 des entêtes répétés sur chaque page  page 1 sur 12  voici un paragraphe avec des tabulations mélangées à des espaces ainsi que des espaces multiples consécutifs ce paragraphe commence par une indentation inutile et contient des

Bonnes pratiques RAG

- Toujours nettoyer AVANT de découper
- Utiliser RecursiveCharacterTextSplitter en priorité
- chunk_size idéal : 1000 à 1200
- overlap entre 10% et 20%
- Éviter les chunks trop courts (< 200 caractères)
- Toujours conserver les métadonnées
- Adapter le chunk_size au format (court pour code • long pour texte narratif)
- Tester plusieurs configurations dans les domaines complexes (juridique, médical, OCR...)
- Vérifier manuellement des chunks au début du projet
- Ne jamais chunker de manière aléatoire ou uniforme sans logique sémantique


Résultats garantis :

- Embeddings plus précis
- Recherche nettement plus pertinente
- Réponses du RAG beaucoup plus fiables